In [5]:
import random
import math
import re

In [6]:
class Lab01:

    def __init__(self):
        self.next_word_counts = {}

    # Принимает список слов обучающего набора
    def fit(self, words):

        for i in range(len(words)-1):
            current_word = words[i]
            next_word = words[i+1]

            if not self.next_word_counts.get(current_word):
                self.next_word_counts[current_word] = {}


            self.next_word_counts[current_word][next_word] = \
                self.next_word_counts[current_word].get(next_word, 0) + 1    

    # Вычисляет вероятность встретить слово w после слова v
    def prob(self, v, w, alpha=0):
        if v not in self.next_word_counts:
            vw_count = 0
            v_count = 0
        else:
            vw_count = self.next_word_counts[v].get(w, 0)
            v_count = sum(self.next_word_counts[v].values())

        if v_count == 0 and alpha == 0:
            return 0

        v_max = len(self.next_word_counts) # Количество уникальных слов, размер словаря

        # В формуле мы используем сглаживание Лапласа, т.к. слов, которых не было в обучающей выборке
        # мы будем получать 0
        return (
            (vw_count + alpha) / (v_count + alpha * v_max)
        )

    # Вычисляет сумму вероятностей, пропущенных через логарифм для заданного текста
    def logprob(self, text: list, alpha=1):

        sum_log_p = 0
        for i in range(len(text)-1):
            
            # Здесь мы передаем параметр сглаживания alpha - чтобы не получать 0 в логарифме
            sum_log_p += math.log(self.prob(text[i], text[i+1], alpha))
        return sum_log_p

    def get_next_word(self, word, T):
        if not self.next_word_counts.get(word):
            print('Нет такого слова')
            return None

        next_words = list(self.next_word_counts[word].keys())
        next_probabilities = []

        for next_word in next_words:
            pr = self.prob(v=word, w=next_word)
            next_probabilities.append(pr**(1/T))

        # Нормализуем вероятности, чтобы их сумма была = 1
        sum_before_normalization = sum(next_probabilities)
        next_probabilities = [x / sum_before_normalization for x in next_probabilities]

        return random.choices(next_words, weights=next_probabilities, k=1)[0]

    # Создает текст с заданной длиной
    def generate(self, word, count_words, T):

        text = word + ' '
        word = word.lower()

        for i in range(count_words):

            next_word = self.get_next_word(word, T)

            if next_word is None:
                break

            text += next_word +  ' '
            word = next_word

        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        return text


## Корпус и токенизация

Обрабатываем исходный текст

In [7]:
with open('C:\Projects\\tgo_course\\lab01\Oblomov.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Переводим в нижний регистр
text = text.lower()

# Ищем слова и выбранные знаки препинания
words = re.findall(r'\w+(?:-\w+)*|[.,!?]', text)

# Делим текст на выборку: 90% train и 10% test
words_test_split = words[int(len(words)*0.9):]
words_train_split = words[:int(len(words)*0.9)]

print(len(words_test_split))
print(len(words_train_split))

18227
164043


Обучаем модель

In [8]:
lab01 = Lab01()

print(words[10000:10020])
lab01.fit(words_train_split)

[',', 'илья', 'ильич', '.', 'куда', 'это', '?', 'да', 'к', 'овчинину-то', ',', 'поедемте', '.', 'там', 'матвей', 'андреич', 'альянов', ',', 'казимир', 'альбертыч']


## Счетчики и вероятности

Смотрим вероятность пары слов на нескольких примерах

In [9]:
print(lab01.prob('может', 'быть'))
print(lab01.prob('завтра', 'будет'))
print(lab01.prob('пойдем', 'к'))
print(lab01.prob('тебе', '?'))

0.6948051948051948
0.010752688172043012
0.037037037037037035
0.052941176470588235


## Генерация

Генерируем 5 предложений по 30 слов с разной температурой T. Можно заметить, что с увеличением T повышается и разнообразие текста - появляются более редки комбинации слов. С уменьшением же T модель генерирует более вероятные слова, текст становится более "сухим"

In [6]:
print('T=1', lab01.generate('кровать', 30, 1))
print('T=2', lab01.generate('кровать', 30, 2))
print('T=4', lab01.generate('кровать', 30, 4))
print('T=0.5', lab01.generate('кровать', 30, 0.5))
print('T=0.25', lab01.generate('кровать', 30, 0.25))

T=1 кровать тюфяки, захар был прочесть сверх того и андрей иваныч дай еще стенные часы и управления крестьянами. верхний этаж назначался для ильи ильича, захар мигал редко туда, 
T=2 кровать тюфяки, опершись на выборы, пятьсот, садилась на троих работает с крапинками, высокенькие, читаете. взял какую-то глупую историю своей сладостной дремоте и попросила ее катя 
T=4 кровать тюфяки, составляющих этот захар деспотически возложил их сквозь сон, обрадовался обломов уселся на бродивших кур и ленивом хождении по необходимости. вечно спертый воздух свежее создание! взгляд 
T=0.5 кровать тюфяки, чтоб не на него. и так, что он, и за дверь, как у него, как только, с ним, что я бы 
T=0.25 кровать тюфяки, что ж, и не было, что я не было. он, и не было легко, что он, что ж ты, и не 


In [7]:
print('T=1', lab01.generate('Дорога', 30, 1))
print('T=2', lab01.generate('Дорога', 30, 2))
print('T=4', lab01.generate('Дорога', 30, 4))
print('T=0.5', lab01.generate('Дорога', 30, 0.5))
print('T=0.25', lab01.generate('Дорога', 30, 0.25))

T=1 Дорога! и женщины а все кое-как, опять замолчат. извергнуть! недавно была русская веру и тебя. она, от лица у ней в этой господствовавшей в каждую 
T=2 Дорога ну что годится, бессовестно! самая мысль на прожиток не замай! никакого не говорила вслух сказал тарантьев спрятал опять ясно в ней цветы, плечи. вечером домой 
T=4 Дорога! отвечала ей вслед в надежде на брошенное в четыре времени с ольгой сергеевной, например да неискусно. поблагодари ее привычках, приглашал обломов, выскочил захарка, вглядываясь 
T=0.5 Дорога, но не кончится ваш братец, так, что на другой день, и не только, и как можно. я не может быть, а то на 
T=0.25 Дорога, что? спросил он. он, как будто с ним, что он, а не было, что он, что?. вот, и не 


In [8]:
print('T=1', lab01.generate('Может', 30, 1))
print('T=2', lab01.generate('Может', 30, 2))
print('T=4', lab01.generate('Может', 30, 4))
print('T=0.5', lab01.generate('Может', 30, 0.5))
print('T=0.25', lab01.generate('Может', 30, 0.25))

T=1 Может быть, нескончаемый разговор, с озабоченным лицом, как обломов стал чертить узор очень хорошо, увядая.. опять на другом. потом она сейчас баб, вдруг 
T=2 Может в солонку. лицо барина зато с вопросом и сухо заметить, отойдет, и она. голиков ли анна андреевна! возразил он давно с деревенскими мальчишками играть с 
T=4 Может проводить дорогу надо деньги брали илья ильич, смотрела рассеянно посмотрел на ходу, такой обидной снисходительностью, староста пополнит бежавших мужиков записывают я звал обломова? приставал штольц давно 
T=0.5 Может быть, сказал он с ольгой. он. не знаю, и по ночам, и не было бы и был в голове. но, как в деревне 
T=0.25 Может быть, и не было, и не было, что ж ты не было, что ж ты не было, а теперь, и не надо было, 


In [9]:
word = 'Есть'
print('T=1', lab01.generate(word, 30, 1))
print('T=2', lab01.generate(word, 30, 2))
print('T=4', lab01.generate(word, 30, 4))
print('T=0.5', lab01.generate(word, 30, 0.5))
print('T=0.25', lab01.generate(word, 30, 0.25))

T=1 Есть обязательная привычка? вдруг точно, и даже, что сказать мне никакого знака, коли не могу оставаться у меня закон не может быть, тихо спросила она и 
T=2 Есть какое-то письмо это трудно отвечать на чем бегать! вот пошлет! видишь ли? от меня у нас, выскочившего к людскому пороку, напирая на шитую подушку головой 
T=4 Есть указывала, сообщительный нрав, думала она примолкла. сыро, счастливая! ему знатность, опечалилась он так внезапно пораженный сном, охватывал его сюда нос утри. фамилию 
T=0.5 Есть, как у него в три. а не с собой, что ж ты сказать, что ли? спросил илья ильич, что? спросила она, как 
T=0.25 Есть, а не было, и не было, а ты не было, что ты, что ж ты, что я не было, как будто в самом 


In [10]:
word = 'будешь'
print('T=1', lab01.generate(word, 30, 1))
print('T=2', lab01.generate(word, 30, 2))
print('T=4', lab01.generate(word, 30, 4))
print('T=0.5', lab01.generate(word, 30, 0.5))
print('T=0.25', lab01.generate(word, 30, 0.25))

T=1 будешь сидеть два смотришь на выборгскую сторону, практической целью посмотреть, чтоб только упорному молчанию. обедаю там сбывают с ильей ильичом. собираются броситься к двери. но, 
T=2 будешь играть с пылающими глазами вот его за коляски платите? здравствуйте, кататься, положив локти не обедали дома сидят в садике ты думал обломов так говорил он принадлежал двум 
T=4 будешь получать с угрозой со стола книгу с будничными лицами, удивляясь, насильственным спокойствием, мети же ловушку, грозно возразил андрей воротился опять сел, всю гостиную штольц продолжал 
T=0.5 будешь играть в доме, а настойчиво, а я не было ни на него в деревню нет, что ты и в это значит, и не видались. он 
T=0.25 будешь жить, и не было, что она. он, что ж ты, что это не то, а я не знаю, а ты не было, 


## Правдоподобие текста

Взяли 50 слов из отложенной части - это осмысленный кусок текста. Затем тот же текст подали в модель в перемешку. Как результат - второй тест показал меньшее правдоподобие

In [10]:
words_for_test = words_test_split[:50]
print(words_for_test)
print(lab01.logprob(words_for_test))

random.shuffle(words_for_test)
print(words_for_test)
print(lab01.logprob(words_for_test))

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе', ',', 'я', 'без', 'того', 'не', 'выйду', 'что', 'вы', ',', 'михей', 'андреич', ',', 'кричите', 'так', '?', 'сказали', 'хозяйка', 'и', 'анисья', ',', 'выглянув', 'из-за', 'дверей', '.', 'двое', 'прохожих', 'остановились', ',', 'слушают', ',']
-439.653921600745
['слушают', '.', ',', ',', '?', 'того', 'вы', 'остановились', 'давай', 'состояния', 'уме', ',', ',', 'из-за', 'давай', 'тебе', 'сказали', 'выглянув', 'что', ',', 'вексель', 'не', 'прохожих', 'его', 'на', 'своем', 'анисья', 'андреич', 'михей', ',', 'имя', 'я', 'не', 'в', ',', 'дверей', 'теперь', 'пьян', 'так', 'двое', 'без', 'и', ',', ',', 'выйду', 'ты', 'половину', 'хозяйка', 'кричите', 'говорю']
-473.1842958120811


Тест работает, т.к. в модели учтено сглаживание Лапласа (см. метод logprob() и prob())

## Оценка

Посчитаем перплексию на отложенном тестовом куске текста

In [11]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
7027.200590121818


Посмотри перплексию с разным параметром сглаживания

In [12]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.1)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
3758.6911735276676


In [13]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.01)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
2895.6819829769133


Видим, что при уменьшении параметра сглаживания перплексия становится лучше

## Дополнительная часть

Попробуем обработать текст без знаков препинания и оценить модель

In [14]:
# Ищем слова без знаков препинания
words = re.findall(r'\w+(?:-\w+)*', text)

# Делим текст на выборку: 90% train и 10% test
words_test_split = words[int(len(words)*0.9):]
words_train_split = words[:int(len(words)*0.9)]

print(len(words_test_split))
print(len(words_train_split))

14794
133143


In [15]:
lab01 = Lab01()

print(words[10000:10020])
lab01.fit(words_train_split)

['а', 'если', 'и', 'примется', 'так', 'не', 'дай', 'бог', 'что', 'выйдет', 'точно', 'ребенок', 'там', 'не', 'доглядит', 'тут', 'не', 'знает', 'каких-нибудь', 'пустяков']


In [17]:
print('T=1', lab01.generate('Может', 30, 1))
print('T=2', lab01.generate('Может', 30, 2))
print('T=4', lab01.generate('Может', 30, 4))
print('T=0.5', lab01.generate('Может', 30, 0.5))
print('T=0.25', lab01.generate('Может', 30, 0.25))

T=1 Может переменить лиловое платье куда же серый сюртук или просто скучно захар воровать не тарантьев зачем же в обморок испытывать ее голосе ее воображение в пяти часам скрестив руки и барана 
T=2 Может и жаркие слезы она нам сейчас сквозь зубы приказчиков мастеровых купцов чиновников с начала мира там ошибка могла противиться обаянию сна ребенок оттого говорил алексеев стараясь одолеть он поддаст захарке 
T=4 Может без этого его язвы и вам что этакого выкопал он вглядывается в первые слова я за весь век живут так искусно умела весь город к грушевой воде к свадьбе и 
T=0.5 Может быть подумал обломов не будет уныло носятся у вас не было не читал книги и только что ж это за что ли ты все это правда ли что ты не 
T=0.25 Может быть и не было бы не было бы не было в доме и не было бы не было бы и не было в самом деле была в доме и не 


Оценим правдоподобие

In [18]:
words_for_test = words_test_split[:50]
print(words_for_test)
print(lab01.logprob(words_for_test))

random.shuffle(words_for_test)
print(words_for_test)
print(lab01.logprob(words_for_test))

['притон', 'шайка', 'мошенников', 'грабят', 'убивают', 'вон', 'мерзавец', 'закричал', 'обломов', 'бледный', 'трясясь', 'от', 'ярости', 'сию', 'минуту', 'чтоб', 'нога', 'твоя', 'здесь', 'не', 'была', 'или', 'я', 'убью', 'тебя', 'как', 'собаку', 'он', 'искал', 'глазами', 'палки', 'батюшки', 'разбой', 'помогите', 'кричал', 'тарантьев', 'захар', 'выбрось', 'вон', 'этого', 'негодяя', 'и', 'чтоб', 'он', 'не', 'смел', 'глаз', 'казать', 'сюда', 'закричал']
-471.2881970704339
['нога', 'тарантьев', 'палки', 'была', 'сюда', 'казать', 'этого', 'не', 'твоя', 'не', 'тебя', 'трясясь', 'глаз', 'как', 'вон', 'захар', 'убью', 'шайка', 'обломов', 'глазами', 'он', 'он', 'вон', 'чтоб', 'помогите', 'убивают', 'выбрось', 'мерзавец', 'батюшки', 'притон', 'или', 'смел', 'и', 'закричал', 'грабят', 'негодяя', 'от', 'сию', 'бледный', 'искал', 'разбой', 'чтоб', 'минуту', 'мошенников', 'ярости', 'я', 'кричал', 'собаку', 'закричал', 'здесь']
-487.816563507846


Посмотрим перплексию

In [19]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['притон', 'шайка', 'мошенников', 'грабят', 'убивают', 'вон', 'мерзавец', 'закричал', 'обломов', 'бледный', 'трясясь', 'от', 'ярости', 'сию', 'минуту', 'чтоб', 'нога', 'твоя', 'здесь', 'не']
12634.578243923379


In [20]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.1)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['притон', 'шайка', 'мошенников', 'грабят', 'убивают', 'вон', 'мерзавец', 'закричал', 'обломов', 'бледный', 'трясясь', 'от', 'ярости', 'сию', 'минуту', 'чтоб', 'нога', 'твоя', 'здесь', 'не']
7636.827256788906


In [21]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.01)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['притон', 'шайка', 'мошенников', 'грабят', 'убивают', 'вон', 'мерзавец', 'закричал', 'обломов', 'бледный', 'трясясь', 'от', 'ярости', 'сию', 'минуту', 'чтоб', 'нога', 'твоя', 'здесь', 'не']
6303.556843427468


Результат: кардинально других результатов мы не получили при отбрасывании знаков препинания

## Работа с триграммами

In [16]:
class Lab01_threegram(Lab01):
    def __init__(self):
        super().__init__()
        self.next_word_counts_threegram = {}

    def fit(self, words):
        super().fit(words)

        for i in range(len(words)-2):
            current_word_1 = words[i]
            current_word_2 = words[i+1]
            key = (current_word_1, current_word_2)
            next_word = words[i+2]

            if not self.next_word_counts_threegram.get(key):
                self.next_word_counts_threegram[key] = {}


            self.next_word_counts_threegram[key][next_word] = \
                self.next_word_counts_threegram[key].get(next_word, 0) + 1  

    # Вычисляет вероятность встретить слово w после слов u и v
    def threegram_prob(self, u, v, w, alpha=0):
        if (u,v) not in self.next_word_counts_threegram:
            return super().prob(v, w, alpha)
        else:
            uvw_count = self.next_word_counts_threegram[(u,v)].get(w, 0)
            uv_count = sum(self.next_word_counts_threegram[(u,v)].values())

        if uv_count == 0 and alpha == 0:
            return 0

        uv_max = len(self.next_word_counts) # Количество уникальных слов, размер словаря, 
                                            # поэтому используем next_word_counts

        # В формуле мы используем сглаживание Лапласа, т.к. для слов, которых не было в обучающей выборке
        # мы будем получать 0
        return (
            (uvw_count + alpha) / (uv_count + alpha * uv_max)
        )

    # Вычисляет сумму вероятностей, пропущенных через логарифм для заданного текста
    def logprob(self, text: list, alpha=1):

        # Высчитываем первый переход как биграмму - для честного сравнения
        sum_log_p = math.log(super().prob(v=text[0], w=text[1], alpha=alpha))        
        
        for i in range(len(text)-2):
            # Здесь мы передаем параметр сглаживания alpha - чтобы не получать 0 в логарифме
            sum_log_p += math.log(self.threegram_prob(u=text[i], v=text[i+1], w=text[i+2], alpha=alpha))
        return sum_log_p

    def get_next_word(self, word_1, word_2, T):
        key = (word_1, word_2)

        if not self.next_word_counts_threegram.get(key)\
            or sum(self.next_word_counts_threegram[key].values()) == 0:
            return super().get_next_word(word_2, T)

        next_words = list(self.next_word_counts_threegram[key].keys())
        next_probabilities = []

        for next_word in next_words:
            pr = self.threegram_prob(u=word_1, v=word_2, w=next_word)
            next_probabilities.append(pr**(1/T))

        # Нормализуем вероятности, чтобы их сумма была = 1
        sum_before_normalization = sum(next_probabilities)
        next_probabilities = [x / sum_before_normalization for x in next_probabilities]

        # print(f'threegram {word_1}, {word_2} ')
        return random.choices(next_words, weights=next_probabilities, k=1)[0]
    
    def generate(self, word_1, word_2, count_words, T):

        word_1, word_2 = word_1.lower(), word_2.lower()
        text = word_1 + ' ' + word_2 + ' '

        for i in range(count_words):

            next_word = self.get_next_word(word_1, word_2, T)

            if next_word is None:
                break

            text += next_word +  ' '
            word_1, word_2 = word_2, next_word

        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        return text      

In [17]:
with open('C:\Projects\\tgo_course\\lab01\Oblomov.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Переводим в нижний регистр
text = text.lower()

# Ищем слова и выбранные знаки препинания
words = re.findall(r'\w+(?:-\w+)*|[.,!?]', text)

# Делим текст на выборку: 90% train и 10% test
words_test_split = words[int(len(words)*0.9):]
words_train_split = words[:int(len(words)*0.9)]

print(len(words_test_split))
print(len(words_train_split))

18227
164043


In [18]:
lab01_threegram = Lab01_threegram()

lab01_threegram.fit(words_train_split)

In [19]:
word_1 = 'кровать'
word_2 = 'стояла'
print(lab01_threegram.generate(word_1, word_2, 30, 1))
print(lab01_threegram.generate(word_1, word_2, 30, 2))
print(lab01_threegram.generate(word_1, word_2, 30, 4))
print(lab01_threegram.generate(word_1, word_2, 30, 0.5))
print(lab01_threegram.generate(word_1, word_2, 30, 0.25))

кровать стояла не убранная от вчерашнего ужина тарелка с солонкой и с другой стороны жизненного опыта, практической истины, было между ними не станет то чувство пробираться так тихо, что 
кровать стояла у фортепьяно и вся вспыхнула, когда как будто не спала, долго освобождался от грубого впечатления. например, продолжал обломов, должно быть! обломов вдруг смолк, 
кровать стояла не убранная от вчерашнего ужина тарелка с солонкой и с ужасом сказал он анисье. а ребенок все наблюдал своим детским, ничего! я очень охотно он взглянул улыбка 
кровать стояла не убранная от вчерашнего ужина тарелка с солонкой и с трудом, мало-помалу, дачи опустели. обломов не видал. зачем же ты это сказал вам, приезжайте-ка обедать 
кровать стояла не убранная от вчерашнего ужина тарелка с солонкой и с трудом, но не мог не сжалиться. ему только семь лет. а он еще не дошла очередь опыта 


In [26]:
word_1 = 'дорога'
word_2 = 'была'
print(lab01_threegram.generate(word_1, word_2, 30, 1))
print(lab01_threegram.generate(word_1, word_2, 30, 2))
print(lab01_threegram.generate(word_1, word_2, 30, 4))
print(lab01_threegram.generate(word_1, word_2, 30, 0.5))
print(lab01_threegram.generate(word_1, word_2, 30, 0.25))

дорога была ее хозяйственным распоряжениям. она пробовала тихонько отнять, но еще свежему лицу. ольга, этот парк, точно одно из чудовищ переходит с своего ложа и чуть не 
дорога была все та же глубокая тишина и мир лежат и замыслы твои служить, торговать, хоть под стол залезть, когда хочет сесть ячмень на глазу, какого блюда и 
дорога была на первом плане всегда грезилась женщина как жена и иногда как любовница. в деревню съездил бы ведь и то, может быть, другое письмо, сколько беготни, 
дорога была в петровки здесь была, возражает илья иванович. смотри-ка, пожалуй, серебряные, даже погорячатся и рассердятся. как я рада, что больше ни ногой. он 
дорога была одна сосновка, другая вавиловка, в самом деле, не подозревая, что он не пошел, едва заметного опыта, практической истины, было в ее глазах удвоилось 


В этих примерах видно, что текст стал чуть более связный, хотя в целом он остается бессмысленным

## Правдоподобие текста

In [20]:
words_for_test = words_test_split[:50]
print(words_for_test)
print(lab01_threegram.logprob(words_for_test))

random.shuffle(words_for_test)
print(words_for_test)
print(lab01_threegram.logprob(words_for_test))

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе', ',', 'я', 'без', 'того', 'не', 'выйду', 'что', 'вы', ',', 'михей', 'андреич', ',', 'кричите', 'так', '?', 'сказали', 'хозяйка', 'и', 'анисья', ',', 'выглянув', 'из-за', 'дверей', '.', 'двое', 'прохожих', 'остановились', ',', 'слушают', ',']
-466.2725958557426
['анисья', 'того', 'я', ',', 'вексель', ',', 'вы', 'не', ',', 'сказали', 'состояния', 'слушают', 'дверей', 'на', ',', 'имя', ',', 'хозяйка', 'не', 'половину', 'давай', 'что', ',', 'ты', 'так', '.', 'теперь', 'без', 'андреич', 'его', 'кричите', 'пьян', 'в', 'давай', ',', 'уме', 'выйду', 'своем', 'и', 'говорю', 'остановились', 'двое', 'выглянув', 'прохожих', 'из-за', ',', ',', 'тебе', 'михей', '?']
-478.35057140943707


In [28]:
words_for_test = words_test_split[:200]
print(words_for_test)
print(lab01_threegram.logprob(words_for_test))

random.shuffle(words_for_test)
print(words_for_test)
print(lab01_threegram.logprob(words_for_test))

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе', ',', 'я', 'без', 'того', 'не', 'выйду', 'что', 'вы', ',', 'михей', 'андреич', ',', 'кричите', 'так', '?', 'сказали', 'хозяйка', 'и', 'анисья', ',', 'выглянув', 'из-за', 'дверей', '.', 'двое', 'прохожих', 'остановились', ',', 'слушают', ',', 'что', 'за', 'крик', 'буду', 'кричать', ',', 'вопил', 'тарантьев', ',', 'пусть', 'срамится', 'этот', 'олух', '!', 'пусть', 'обдует', 'тебя', 'этот', 'мошенник', 'немец', ',', 'благо', 'он', 'теперь', 'стакнулся', 'с', 'твоей', 'любовницей', 'в', 'комнате', 'раздалась', 'громкая', 'оплеуха', '.', 'пораженный', 'обломовым', 'в', 'щеку', ',', 'тарантьев', 'мгновенно', 'смолк', ',', 'опустился', 'на', 'стул', 'и', 'в', 'изумлении', 'ворочал', 'вокруг', 'одуревшими', 'глазами', '.', 'что', 'это', '?', 'что', 'это', 'а', '?', 'что', 'это', '!', 'бледный', ',', 'задыхаясь', ',', 'говорил', 'он', ',',

В первом тесте правдободобие показало примерно одинаковый результат - при разных запусках значения примерно одинаковы. Связано это может быть с тем, что для триграммной модели 50 слов - это слишком мало токенов. Видно, что при увеличении слов до 200 правдоподобие у настоящего текста стабильно выше, чем у случайного

## Оценка

Посчитаем перплексию на отложенном тестовом куске текста

In [21]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01_threegram.logprob(words_for_test)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
12330.039780725017


Посмотри перплексию с разным параметром сглаживания

In [30]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01_threegram.logprob(words_for_test, alpha=0.1)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
7269.842560786511


In [31]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01_threegram.logprob(words_for_test, alpha=0.01)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
4940.891203028781


По сравнению с биграммами общий показатель перплексии стал хуже. Самое лучшее значение также получено при alpha=0.01

# Работа с Qwen

In [1]:
import os, math, torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

plt.rcParams.update({"figure.dpi": 110, "font.size": 12})
os.makedirs("fallback", exist_ok=True)

MODEL_ID = os.environ.get("TGO_MODEL_ID", "Qwen/Qwen3-0.6B-Base")

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32
    )  # transformers 4.x
model = model.to(device).eval()
print(f"модель: {MODEL_ID}, параметров: {sum(p.numel() for p in model.parameters())/1e6:.0f} млн, словарь: {len(tok)} токенов")

c:\Projects\tgo_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 320.07it/s]


модель: Qwen/Qwen3-0.6B-Base, параметров: 596 млн, словарь: 151669 токенов


In [30]:
@torch.no_grad()
def sequence_log_likelihood(text: str):
    ids = tok(text, return_tensors="pt").to(device)["input_ids"]
    logits = model(ids).logits[0, :-1].float() # предсказания для позиций 1..T
    logp = torch.log_softmax(logits, dim=-1)
    target = ids[0, 1:]
    token_logp = logp[torch.arange(target.numel()), target].cpu()
    return token_logp.sum().item(), token_logp.numel()

texts = {
    "связный текст": "Москва — столица России и крупнейший город страны.",
    "те же слова вперемешку": "Страны город и — крупнейший Москва столица России.",
}
for name, t in texts.items():
    ll, n = sequence_log_likelihood(t)
    print(f"{name:<26} log P = {ll:8.1f}   (на токен: {ll/n:6.2f},  perplexity: {math.exp(-ll/n):8.1f})")

связный текст              log P =    -21.9   (на токен:  -1.57,  perplexity:      4.8)
те же слова вперемешку     log P =    -64.0   (на токен:  -4.57,  perplexity:     96.8)


In [23]:
print(words_test_split[:50])

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе', ',', 'я', 'без', 'того', 'не', 'выйду', 'что', 'вы', ',', 'михей', 'андреич', ',', 'кричите', 'так', '?', 'сказали', 'хозяйка', 'и', 'анисья', ',', 'выглянув', 'из-за', 'дверей', '.', 'двое', 'прохожих', 'остановились', ',', 'слушают', ',']


In [25]:
words_for_test = words_test_split[:50].copy()

words_for_test_random = words_for_test.copy()
random.Random(42).shuffle(words_for_test_random)

# Переводим в текст для Qwen
text_for_test = re.sub(
    r'\s+([.,!?;:])',
    r'\1',
    ' '.join(words_for_test)
)

text_for_test_random = re.sub(
    r'\s+([.,!?;:])',
    r'\1',
    ' '.join(words_for_test_random)
)

print(text_for_test)
print(text_for_test_random)

половину состояния давай вексель на его имя ты теперь не пьян, в своем уме, давай, говорю тебе, я без того не выйду что вы, михей андреич, кричите так? сказали хозяйка и анисья, выглянув из-за дверей. двое прохожих остановились, слушают,
выйду того тебе, на прохожих что не михей давай, я в вексель, анисья пьян не сказали половину. говорю так слушают из-за андреич,, без дверей остановились хозяйка кричите двое своем,, давай вы и его? имя теперь уме,, состояния ты выглянув


In [26]:
print(words_train_split[:30])

['обломов', 'автор', 'иван', 'александрович', 'гончаров', 'экспортировано', 'из', 'викитеки', '30', 'сентября', '2026', 'г', '.', 'часть', 'i', 'глава', 'i', 'глава', 'ii', 'глава', 'iii', 'глава', 'iv', 'глава', 'v', 'глава', 'vi', 'глава', 'vii', 'глава']


Переобучаем модели

In [27]:
lab01 = Lab01()
lab01.fit(words_train_split)

lab01_threegram = Lab01_threegram()
lab01_threegram.fit(words_train_split)

In [31]:
alpha = 0.01
N = len(words_for_test) - 1

# Биграммы
bigram_logp = lab01.logprob(words_for_test, alpha)
bigram_random_logp = lab01.logprob(words_for_test_random, alpha)

# Триграммы
threegram_logp = lab01_threegram.logprob(words_for_test, alpha)
threegram_random_logp = lab01_threegram.logprob(
    words_for_test_random,
    alpha
)

# Qwen
qwen_logp, qwen_N = sequence_log_likelihood(text_for_test)
qwen_random_logp, qwen_random_N = sequence_log_likelihood(
    text_for_test_random
)

print('Биграммы')
print('Связный:', bigram_logp, math.exp(-bigram_logp / N))
print('Перемешанный:', bigram_random_logp, math.exp(-bigram_random_logp / N))
print('Разрыв:', bigram_logp - bigram_random_logp)

print('\nТриграммы')
print('Связный:', threegram_logp, math.exp(-threegram_logp / N))
print('Перемешанный:', threegram_random_logp, math.exp(-threegram_random_logp / N))
print('Разрыв:', threegram_logp - threegram_random_logp)

print('\nQwen')
print('Связный:', qwen_logp, math.exp(-qwen_logp / qwen_N))
print(
    'Перемешанный:',
    qwen_random_logp,
    math.exp(-qwen_random_logp / qwen_random_N)
)
print('Разрыв:', qwen_logp - qwen_random_logp)

Биграммы
Связный: -372.1696725301048 1988.825294979057
Перемешанный: -442.4146405445024 8340.431354629756
Разрыв: 70.24496801439761

Триграммы
Связный: -397.48920519763357 3334.3218550873166
Перемешанный: -438.119460470322 7640.4629737747955
Разрыв: 40.63025527268843

Qwen
Связный: -349.00030517578125 52.768401694684094
Перемешанный: -457.6997375488281 248.25455948520633
Разрыв: 108.69943237304688


Все три модели оценили связный текст выше перемешанного: его логарифмическая вероятность оказалась больше, а перплексия — меньше. Наибольший разрыв в logP показал Qwen, что означает более уверенное распознавание настоящего текста, а не случайного